In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [3]:
df_green = spark.read.parquet('data/pq/green/*/*')

In [4]:
df_green.createOrReplaceTempView("green")

In [5]:
df_green_revenue = spark.sql(
    """
    SELECT 
    date_trunc("hour", lpep_pickup_datetime) AS hour,
    PULocationID AS zone, 
    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
    FROM green
    WHERE lpep_pickup_datetime >= '2020-01-01 00:00:00'
    GROUP BY 1,2
    ORDER BY 1,2
    """
)

In [6]:
df_green_revenue.show()

+-------------------+----+------------------+--------------+
|               hour|zone|            amount|number_records|
+-------------------+----+------------------+--------------+
|2020-01-01 00:00:00|   7| 769.7299999999996|            45|
|2020-01-01 00:00:00|  17|195.03000000000003|             9|
|2020-01-01 00:00:00|  18|               7.8|             1|
|2020-01-01 00:00:00|  22|              15.8|             1|
|2020-01-01 00:00:00|  24|              87.6|             3|
|2020-01-01 00:00:00|  25|             531.0|            26|
|2020-01-01 00:00:00|  29|              61.3|             1|
|2020-01-01 00:00:00|  32| 68.94999999999999|             2|
|2020-01-01 00:00:00|  33|317.27000000000004|            11|
|2020-01-01 00:00:00|  35|            129.96|             5|
|2020-01-01 00:00:00|  36|295.34000000000003|            11|
|2020-01-01 00:00:00|  37|            175.67|             6|
|2020-01-01 00:00:00|  38| 98.78999999999999|             2|
|2020-01-01 00:00:00|  4

In [7]:
df_green_revenue \
    .repartition(20) \
    .write.parquet("data/report/revenue/green", mode='overwrite')

In [8]:
df_yellow = spark.read.parquet('data/pq/yellow/*/*')
df_yellow.createOrReplaceTempView("yellow")

df_yellow_revenue = spark.sql(
    """
    SELECT 
    date_trunc("hour", tpep_pickup_datetime) AS hour,
    PULocationID AS zone, 
    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
    FROM yellow
    WHERE tpep_pickup_datetime >= '2020-01-01 00:00:00'
    GROUP BY 1,2
    ORDER BY 1,2
    """
)
df_yellow_revenue.show()

+-------------------+----+------------------+--------------+
|               hour|zone|            amount|number_records|
+-------------------+----+------------------+--------------+
|2020-01-01 00:00:00|   3|              25.0|             1|
|2020-01-01 00:00:00|   4|1004.3000000000003|            57|
|2020-01-01 00:00:00|   7|455.17000000000013|            38|
|2020-01-01 00:00:00|  10|             42.41|             2|
|2020-01-01 00:00:00|  12|106.99999999999999|             6|
|2020-01-01 00:00:00|  13|1214.8000000000002|            56|
|2020-01-01 00:00:00|  14|               8.8|             1|
|2020-01-01 00:00:00|  15|             34.09|             1|
|2020-01-01 00:00:00|  17|220.20999999999998|             8|
|2020-01-01 00:00:00|  18|               5.8|             1|
|2020-01-01 00:00:00|  24| 754.9500000000002|            45|
|2020-01-01 00:00:00|  25|            324.35|            16|
|2020-01-01 00:00:00|  32|              18.0|             1|
|2020-01-01 00:00:00|  3

In [9]:
df_yellow_revenue \
    .repartition(20) \
    .write.parquet("data/report/revenue/yellow", mode='overwrite')

In [16]:
df_green_revenue = spark.read.parquet('data/report/revenue/green')

In [17]:
df_yellow_revenue = spark.read.parquet('data/report/revenue/yellow')

In [18]:
df_green_revenue_tmp = df_green_revenue \
    .withColumnRenamed("amount", "green_amount") \
    .withColumnRenamed("number_records", "green_number_records")

df_yellow_revenue_tmp = df_yellow_revenue \
    .withColumnRenamed("amount", "yellow_amount") \
    .withColumnRenamed("number_records", "yellow_number_records")

In [19]:
df_join = df_green_revenue_tmp.join(df_yellow_revenue_tmp, on=['hour', 'zone'], how='outer')

In [20]:
df_join.write.parquet('data/report/revenue/total', mode='overwrite')

In [22]:
df_join = spark.read.parquet('data/report/revenue/total')

In [24]:
df_zones = spark.read.parquet("zones")

In [26]:
df_result = df_join.join(df_zones, df_join.zone == df_zones.LocationID)

In [30]:
df_result

DataFrame[hour: timestamp, zone: int, green_amount: double, green_number_records: bigint, yellow_amount: double, yellow_number_records: bigint, LocationID: string, Borough: string, Zone: string, service_zone: string]

In [31]:
df_result.drop("LocationID", "zone").write.parquet("tmp/revenue-zones")